# 04 · Generador de ruido (jitter)

Baseline exigido por el enunciado: datos reales perturbados con ruido gaussiano. Incluye el barrido de sigma que justifica el valor adoptado.

**Entradas**

- `data/processed/ventanas.npz`

**Salidas**

- `models/generadores/jitter/ (modelo.pkl o .keras, historial.csv, meta.json)`
- `data/synthetic/jitter.npz`
- `results/figures/barrido_sigma_jitter.png`
- `results/metricas/barrido_sigma_jitter.csv`

**Tiempo estimado:** ~6 min en CPU (el barrido de sigma domina; el ajuste en sí es instantáneo).

**Independencia.** Este notebook solo lee `data/processed/ventanas.npz` (notebook 02) y solo escribe en `models/generadores/jitter/` y `data/synthetic/jitter.npz`. No depende de ningún otro notebook de generador ni de sus salidas, de modo que los notebooks 04 a 10 pueden ejecutarse en paralelo y en cualquier orden por distintas personas.

In [ ]:
import sys; sys.path.insert(0, "..")   # permite ejecutar desde notebooks/
import src                              # fija el backend de Keras a PyTorch
from src import config, viz
config.fijar_semillas()
viz.aplicar_estilo()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import ventanas

part = ventanas.cargar_procesado()
train, val, test = part.train, part.val, part.test
print(train, val, test, sep="\n")

In [ ]:
from src import regimenes
from src.generadores import base

v = config.ventanas()
n_regimenes = config.n_regimenes()

bloque_train = ventanas.empaquetar(train)
print("bloque de train:", bloque_train.shape, "· d esperada:", ventanas.dimension_bloque(v))
regimenes.distribucion(train.y_reg, n_regimenes)

## Qué hace y qué no hace

El jitter **no aprende una distribución**: perturba muestras existentes. Genera
puntos en una bola alrededor de cada dato real, así que solo interpola localmente y
nunca produce una configuración de mercado que no se pareciera ya a una observada.

Es exactamente por eso que sirve de suelo. Mide cuánto del beneficio de los datos
sintéticos viene de una regularización barata por ruido y cuánto de haber modelado
de verdad la distribución conjunta. En la literatura de aumento de datos los
baselines triviales ganan a menudo; si ningún generador neuronal bate al jitter,
ese resultado negativo es tan reportable como el positivo.

El ruido es **relativo a la desviación de cada dimensión**, no absoluto: el bloque
mezcla retornos diarios y z-scores cuyas escalas difieren en órdenes de magnitud, y
un sigma absoluto ahogaría unos canales y no tocaría otros.

## Barrido de sigma

Sigma es el único hiperparámetro y tiene un compromiso claro:

- por debajo de ~0.05 las muestras son copias casi exactas de los datos reales: el
  dataset crece pero no aporta información nueva;
- por encima de ~0.5 el ruido destruye la estructura temporal y las correlaciones
  entre canales, y las muestras dejan de parecer mercado.

El barrido mide dos cosas opuestas: cuánto copia y cuánto destruye. El cociente
DVMC por debajo de 1 significa que los sintéticos están más cerca de los reales de
lo que los reales están entre sí, es decir, que hay copia. El error de
correlaciones entre canales crece cuando el ruido rompe la estructura del panel.

La distancia se mide en el espacio proyectado por PCA, no en las 1.201 dimensiones
originales: con tan pocas muestras por dimensión todas las distancias entre pares
convergen al mismo valor y el vecino más cercano deja de significar nada. El
proyector se ajusta una sola vez con los reales y se reutiliza en todo el barrido,
para que los cinco valores de sigma se juzguen en el mismo espacio.

In [ ]:
from src import evaluacion

SIGMAS = (0.02, 0.05, 0.10, 0.20, 0.50)
pca = evaluacion.proyector(bloque_train, semilla=config.semilla())
filas = []

for s in SIGMAS:
    tanteo = base.instanciar("jitter", n_regimenes=n_regimenes, sigma=s)
    tanteo.fit(bloque_train, train.y_reg)
    sint, _ = tanteo.generate_dataset({k: 800 for k in range(n_regimenes)})

    fila = {"sigma": s}
    fila.update(evaluacion.distancia_vecino_mas_cercano(bloque_train, sint, muestra=800, pca=pca))
    fila["error_correlaciones"] = evaluacion.error_correlaciones(
        bloque_train, sint, v.pasado, config.n_canales()
    )
    fila.update(evaluacion.error_autocorrelacion(
        bloque_train, sint, v.pasado, config.n_canales()
    ))
    fila["dif_curtosis"] = float(
        evaluacion.comparar_momentos(bloque_train, sint).loc["curtosis", "diferencia"]
    )
    filas.append(fila)

barrido = evaluacion.acumular(filas, "barrido_sigma_jitter")
barrido.round(4)

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 4))

ejes[0].plot(barrido["sigma"], barrido["cociente_dvmc"], marker="o", color=viz.color("jitter"))
ejes[0].axhline(1.0, color=viz.TINTA_SECUNDARIA, linestyle="--", linewidth=1.2,
                label="distancia típica entre reales")
ejes[0].set_xscale("log")
ejes[0].set_xlabel("sigma (fracción de desviación típica)")
ejes[0].set_ylabel("cociente DVMC")
ejes[0].set_title("Memorización")
ejes[0].legend(fontsize=8)

ejes[1].plot(barrido["sigma"], barrido["error_correlaciones"], marker="o",
             color=viz.color("jitter"), label="correlaciones entre canales")
ejes[1].plot(barrido["sigma"], barrido["error_acf_absolutos"], marker="s",
             color=viz.PALETA[1], label="ACF de |retornos|")
ejes[1].set_xscale("log")
ejes[1].set_xlabel("sigma (fracción de desviación típica)")
ejes[1].set_ylabel("error")
ejes[1].set_title("Destrucción de la estructura")
ejes[1].legend(fontsize=8)

fig.tight_layout()
viz.guardar(fig, "barrido_sigma_jitter")

## Valor adoptado

`sigma = 0.10`. Una décima de desviación típica sitúa el ruido por debajo de la
variabilidad natural de los datos sin ser despreciable: el cociente DVMC se separa
de cero —no hay copia literal— mientras el error de correlaciones y el de la
autocorrelación de los valores absolutos siguen siendo pequeños. El valor queda
documentado aquí y no se vuelve a tocar.

In [ ]:
SIGMA = 0.10

generador = base.instanciar("jitter", n_regimenes=n_regimenes, sigma=SIGMA)
generador.fit(bloque_train, train.y_reg)
generador

## Convergencia

El jitter no optimiza nada, así que no hay curva de pérdida. El diagnóstico
equivalente es el registro por régimen: cuántas muestras reales hay disponibles
para perturbar y cuánto se aleja una muestra perturbada de su original, en unidades
de desviación típica. El eje horizontal es el índice de régimen, no la época.

La lectura relevante es el número de muestras del régimen de crisis: el jitter no
puede inventar una clase que no vio, y con pocos originales su banco sintético es
poco diverso por construcción.

In [ ]:
fig, eje = plt.subplots()
viz.curva_convergencia(
    generador.historial.drop(columns="regimen"),
    "Diagnóstico · " + generador.etiqueta,
    eje=eje,
)
eje.set_xlabel("régimen")
eje.set_yscale("log")
viz.guardar(fig, "convergencia_jitter")

generador.historial

## Inspección visual

Proyección PCA de reales y sintéticos, con la PCA ajustada **solo con los reales**
para que los ejes describan la estructura del mercado y no la del generador.

Es la comprobación más rápida y la que detecta los dos fallos gruesos: si la nube
sintética no cubre la real, el generador ha colapsado a un modo; si la desborda
ampliamente, está inventando configuraciones de mercado que nunca ocurrieron.

Se mira el régimen de crisis porque es el que tiene menos datos reales y, por
tanto, donde el generador tiene más margen para desviarse.

In [ ]:
CRISIS = n_regimenes - 1

muestra_crisis = generador.generate(600, regimen=CRISIS)
reales_crisis = bloque_train[train.y_reg == CRISIS]

fig, ejes = plt.subplots(1, 2, figsize=(12, 4.5))
viz.real_vs_sintetico(bloque_train, generador.generate(600, regimen=0),
                      "{} · régimen de calma".format(generador.etiqueta), eje=ejes[0])
viz.real_vs_sintetico(reales_crisis, muestra_crisis,
                      "{} · régimen de crisis".format(generador.etiqueta), eje=ejes[1])
fig.tight_layout()
viz.guardar(fig, "pca_" + generador.nombre)

print("reales de crisis:", len(reales_crisis), "· sintéticos generados:", len(muestra_crisis))

## Banco de muestras

Se genera un banco uniforme por régimen y se exporta a `data/synthetic/`. La mezcla
concreta de cada dataset la decide el notebook 11 muestreando de este banco, no
volviendo a invocar al generador: así el barrido no necesita tener los siete
modelos cargados en memoria y dos ejecuciones del notebook 12 usan exactamente las
mismas muestras sintéticas.

El banco es uniforme —no replica el desbalance real— porque la política de reparto
es un grado de libertad del experimento y se aplica después.

In [ ]:
MUESTRAS_POR_REGIMEN = 4000

reparto = {k: MUESTRAS_POR_REGIMEN for k in range(n_regimenes)}
bloques_sint, y_sint = generador.generate_dataset(reparto)

print("banco:", bloques_sint.shape, "· etiquetas:", np.bincount(y_sint, minlength=n_regimenes))
print("rango de valores:", round(float(bloques_sint.min()), 2), "→",
      round(float(bloques_sint.max()), 2),
      "(referencia real:", round(float(bloque_train.min()), 2), "→",
      round(float(bloque_train.max()), 2), ")")

## Persistencia

`guardar()` deja el modelo, la curva de convergencia y los metadatos en
`models/generadores/`. Es lo que permite que el resto del grupo salte directamente
al análisis sin reentrenar nada.

In [ ]:
ruta_muestras = generador.exportar_muestras(bloques_sint, y_sint)
ruta_modelo = generador.guardar()

print("muestras:", ruta_muestras)
print("modelo:  ", ruta_modelo)
pd.Series(generador.resumen_convergencia()).round(4)

## Salidas generadas

In [ ]:
from pathlib import Path

salidas = [
    src.DIR_MODELOS_GEN / "jitter" / "meta.json",
    src.DIR_MODELOS_GEN / "jitter" / "historial.csv",
    src.DIR_SINTETICO / "jitter.npz",
    src.DIR_FIGURAS / "convergencia_jitter.png",
    src.DIR_FIGURAS / "pca_jitter.png",
    src.DIR_FIGURAS / "barrido_sigma_jitter.png",
    src.DIR_METRICAS / "barrido_sigma_jitter.csv",
]

for ruta in salidas:
    ruta = Path(ruta)
    marca = "ok" if ruta.exists() else "--"
    print("[{}] {}".format(marca, ruta.relative_to(src.RAIZ)))
